# Forms — From `Questionnaire` to a FHIR `O21` Order

This is the eleventh notebook in the series. Forms for orders and referrals are
common in NHS Trust EPRs — and just as common on the *report* side, where HL7's
[Laboratory Results Interface (LRI)](https://www.hl7.org/implement/standards/product_brief.cfm?product_id=279)
(the HL7 v2 counterpart to FHIR's Genomics Reporting IG) uses the same "panel" concept
this repo's own `05-test-results-from-vcf.ipynb`/`06-eu-laboratory-report-fhir-document.ipynb`
build against — a set of profiled `Observation`s grouped under one report — and pairs
it with a FHIR `Questionnaire` so that same v2 LRI panel can be rendered and completed
as a form. A general-purpose FHIR form renderer such as the National Library of
Medicine's [LHC-Forms](https://lhcforms.nlm.nih.gov/lhcforms) — built for exactly this
kind of `Questionnaire`-in, `QuestionnaireResponse`-out workflow — is the closest
public, runnable demo of what that looks like in practice; NW-GMSA doesn't publish its
own hosted demo of this specific pairing, but any standard R4 `Questionnaire`,
including the one this notebook uses, should render in it directly.

[nw-gmsa.github.io/en/ServiceRequest.html](https://nw-gmsa.github.io/en/ServiceRequest.html)
lists NW-GMSA's own order-side `Questionnaire`s, aimed at the EPR application
specialists who configure order-entry forms within a Trust's EPR — a common core
(`Questionnaire-GenomicTestOrder`, `Questionnaire-GenomicGeneralAskAtOrderEntry`) plus
per-test-type extensions
([`Questionnaire-dWGSSubOrder`](https://nw-gmsa.github.io/en/Questionnaire-dWGSSubOrder.html),
[`Questionnaire-HistocompatibilityAskAtOrderEntry`](https://nw-gmsa.github.io/en/Questionnaire-HistocompatibilityAskAtOrderEntry.html) —
the latter the same H&I use case `10-histocompatibility-immunogenetics-hl7v2-nw-standard.ipynb`
covers). Once submitted, a completed form typically becomes an `ORM_O01`/`OML_O21`
order — this notebook builds the FHIR `O21` Message `Bundle` end of that, starting
from `Questionnaire-dWGSSubOrder` and a real completed
`QuestionnaireResponse` for it, and closes by explaining how the same extracted
answers become an HL7 v2 `O21` instead.

## The `LAB-1` process flow

[nw-gmsa.github.io/en/LTW.html#lab-1-process-flow](https://nw-gmsa.github.io/en/LTW.html#lab-1-process-flow)
lays out the business steps behind this: **select test form** → **complete form** →
**form data extraction** → **submit order** (specimen collection runs in parallel, once
a clinician/nurse actually takes the sample). Technically: the Order Placer (EPR) can
submit either straight to HL7 v2 (`O01`/`O21`, for an EPR with no FHIR capability) or as
an HL7 FHIR Message `O21` — this notebook's route — to the RIE, which then produces the
HL7 v2 `OML_O21` message the destination LIMS (Order Filler) actually needs, the same
canonical-model/message-routing pattern `10-histocompatibility-immunogenetics-hl7v2-nw-standard.ipynb`
covers for H&I.

![notebook 11 flow](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgICBBWyJDbGluaWNpYW4gc2VsZWN0cyBhXG50ZXN0IGZvcm1cbihGSElSIFF1ZXN0aW9ubmFpcmUpIl0gLS0+IEJbIkNvbXBsZXRlcyB0aGUgZm9ybVxuKEZISVIgUXVlc3Rpb25uYWlyZVJlc3BvbnNlKSJdCiAgICBCIC0tPiBDWyJFeHRyYWN0aW9uXG4oZGVmaW5pdGlvbi1iYXNlZCArXG5PYnNlcnZhdGlvbi1iYXNlZCkiXQogICAgQyAtLT4gRFsiUGF0aWVudCAvIFNwZWNpbWVuIC9cblNlcnZpY2VSZXF1ZXN0IC8gT2JzZXJ2YXRpb24iXQogICAgRCAtLT4gRVsiRkhJUiBNZXNzYWdlIE8yMSBCdW5kbGUiXQogICAgRSAtLT58IkhMNyBGSElSIE1lc3NhZ2UgTzIxInwgUklFWyJSZWdpb25hbCBJbnRlZ3JhdGlvblxuRW5naW5lIChSSUUpIl0KICAgIEEyWyJFUFIgd2l0aG91dCBGSElSIHN1cHBvcnQiXSAtLi0+fCJITDcgdjIgTzAxL08yMSJ8IFJJRQogICAgUklFIC0tPnwiSEw3IHYyIE9NTF9PMjEifCBMSU1TWyJMSU1TXG4oT3JkZXIgRmlsbGVyKSJdCg==)


In [1]:
import json
import os
from datetime import datetime
from uuid import uuid4

import requests
from dotenv import load_dotenv

load_dotenv()

toolsServer = os.getenv("V2_TOOLS")  # /transformToV2, used once at the end - optional, needs the lab network

NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
ODS_SYSTEM = "https://fhir.nhs.uk/Id/ods-organization-code"
V2_0203 = "http://terminology.hl7.org/CodeSystem/v2-0203"
GENOMICS_ENGLAND_ODS = "8J834"   # runs NGIS - assigner of patient_ngis_id
NW_GLH_ODS = "699X0"             # NW Genomics - this order's Order Filler
NEY_GENOMICS_ODS = "699N0"       # NE&Y Genomics (Northern Genetics Service, Newcastle) - this order's Order Placer

QUESTIONNAIRE_URL = "https://nw-gmsa.github.io/en/Questionnaire-dWGSSubOrder.json"
QUESTIONNAIRE_RESPONSE_URL = "https://nw-gmsa.github.io/en/QuestionnaireResponse-dWGS-Duo-r2026000202-p2026000102.json"

questionnaire = requests.get(QUESTIONNAIRE_URL).json()
questionnaire_response = requests.get(QUESTIONNAIRE_RESPONSE_URL).json()

print(questionnaire["title"])
print(questionnaire_response["questionnaire"])
print("subject:", questionnaire_response["subject"]["display"],
      "NHS number", questionnaire_response["subject"]["identifier"]["value"])

North West Genomics dWGS Sub-Order Manifest
https://fhir.nwgenomics.nhs.uk/Questionnaire/dWGSSubOrder
subject: Allanys Middlesborough NHS number 9737873971


## The form: `Questionnaire/dWGSSubOrder`

Live-fetched above, not vendored — the same `.json` a form renderer or the RIE's own
tooling would request. It captures a **dWGS sub-contracted order**: the same `LAB-35`
sub-order `08-subcontracted-laboratory-order-from-external-glh.ipynb` builds from a CSV
manifest, this time as an actual order-entry/digital-manifest *form*. Six groups, in
order: `Referral` (the order itself), `Patient`, `AskAtOrderEntry` (Family
Structure/Participant Type — the Duo/Trio questions `08` also asks), `PrimarySpecimen`
("as received by GLH" — filled at referral time), and two groups only the receiving
GLH itself can complete: `DispatchedSpecimen` ("extracted DNA sent onward") and
`LaboratoryQC` (concentration, purity, DIN, dispatch date, rack position). This
`Questionnaire` doubles as the referring Trust's order form *and* NW Genomics' own
digital manifest — two different organisations complete different parts of the same
form, not one clinician filling in QC results they'd have no way of knowing.

In [2]:
def walk_questionnaire(items, out, indent=0):
    for item in items:
        out[item["linkId"]] = {"definition": item.get("definition"), "code": item.get("code"), "type": item["type"]}
        if item["type"] != "group":
            marker = " (code: %s#%s)" % (item["code"][0]["system"], item["code"][0]["code"]) if item.get("code") else ""
            print(f"{'  ' * indent}{item['linkId']:45} {item['type']:8} {item['text']}{marker}")
        if item.get("item"):
            walk_questionnaire(item["item"], out, indent + 1)


definitions = {}
walk_questionnaire(questionnaire["item"], definitions)

  dWGS/referral_id                              string   Original Order Placer Group Number (Referral ID)
  dWGS/clinical_indication_test_type_id         choice   Test Code
  dWGS/ordering_entity_id                       string   Original Ordering Facility Code
    dWGS/ordering_entity_id-designNote            display  ODS code of the original referring Trust - assigner of the Received Sample Identifier (type=PLAC) below, not the same as the Filler Order Ordering Facility Code.
  dWGS/glh_laboratory_id                        string   Filler Order Ordering Facility Code (GLH)
    dWGS/glh_laboratory_id-designNote             display  A Genomic Laboratory Hub (GLH) ODS code, not the referring Trust's own ODS code - also assigns ServiceRequest.requisition and the Specimen's LIMS identifier (type=FILL).
  dWGS/retrospective_sample                     string   Retrospective Sample Flag
    dWGS/retrospective_sample-designNote          display  Values: "New" (ServiceRequest.intent = filler-o

### Why some questions are coded and others aren't

Not every `item.code` above is the same kind of thing. `LN/45392-8`, `LN/21112-8`,
`LN/66746-9` and the rest of the `LN/*` items are [LOINC](https://loinc.org) codes;
`NOS/FamilyStructure`/`NOS/ParticipantType` use NW-GMSA's own `NWGMSA` `CodeSystem`;
most `dWGS/*` items (`dWGS/referral_id`, `dWGS/glh_laboratory_id`, ...) have no
`item.code` at all. That split isn't arbitrary - it's what came out of actually
elaborating this `Questionnaire`: local codes were ruled out early, because a
question's *meaning* has to be unambiguous across every EPR and LIMS that might send
or receive it, not just within the system that first defined it - a local code only
means something to the system that minted it. The search order from there was SNOMED
CT first, then LOINC where SNOMED had nothing suitable - which, in practice, was
often, because LOINC has modelled questions-as-panels for far longer than FHIR
`Questionnaire` has existed (its panel/survey concept predates FHIR entirely), so a
LOINC code already exists for a lot of ordinary demographic and specimen questions
that SNOMED CT was never trying to code in the first place. `dWGS/*` items with no
`item.code` are the residue: genuinely local concepts (a referral ID, a rack well
position) that neither terminology has any reason to model - though not necessarily
permanently: `Questionnaire-dWGSSubOrder` is still being elaborated at the time of
writing, and some of today's uncoded `dWGS/*` items may yet turn out to have a
SNOMED CT or LOINC code after all, the same way `NOS/FamilyStructure`/
`NOS/ParticipantType` gained the `NWGMSA` codes noted above.

This isn't a FHIR-specific problem, either - the same idea (give each question in a
form its own stable, shared code rather than a local one) is exactly what openEHR's
Templates and Archetypes do, independently of FHIR.

### Pre-populating the form

[build.fhir.org/ig/HL7/sdc/en/populate.html](https://build.fhir.org/ig/HL7/sdc/en/populate.html) —
before a clinician sees this form at all, a renderer can pre-fill it from data the EPR
already holds, rather than asking for it again. Two mechanisms: an explicit `$populate`
operation that returns a partially-filled `QuestionnaireResponse` up front, or
*continuous population*, where the form filler resolves answers as the user works
through it (useful when an earlier answer changes what a later one should default to).
Underneath either one, population can be **observation-based** (look up an existing
`Observation` matching a question's code), **expression-based** (`initialExpression`/
`candidateExpression` FHIRPath or CQL against the EPR's own data), or, for the most
complex cases, **`StructureMap`-based**.

Concretely, for this form: `Patient` group answers (name, DOB, NHS number) are exactly
what a referring Trust's EPR already holds on the patient placing the order — classic
automated population, no clinician input needed. The `DispatchedSpecimen`/
`LaboratoryQC` groups are the opposite case: nothing to pre-populate from *when the
referring Trust first submits* the form, because that data doesn't exist yet — it's
NW Genomics' own LIMS that populates (and completes) those groups later, once the
specimen has actually arrived and been processed.

## The completed form: `QuestionnaireResponse/dWGS-Duo-r2026000202-p2026000102`

Also live-fetched above — a real completed response to the `Questionnaire` above, for
the same referral `08-subcontracted-laboratory-order-from-external-glh.ipynb` built
from `Input/dWGS.csv` row 1 (`r2026000202`/`p2026000102`, Allanys Middlesborough) as
[`Input/FHIR/O21/dWGS_r2026000202_p2026000102.json`](https://github.com/nw-gmsa/Testing/blob/main/Input/FHIR/O21/dWGS_r2026000202_p2026000102.json) —
same order, two different starting points: a CSV row there, a filled-in form here.

In [3]:
def walk_response(items, out, indent=0):
    for item in items:
        answer = (item.get("answer") or [None])[0]
        if answer is not None:
            value_key = next(k for k in answer if k.startswith("value"))
            value = answer[value_key]
            out[item["linkId"]] = value
            print(f"{'  ' * indent}{item['linkId']:45} {str(value)[:70]}")
        if item.get("item"):
            walk_response(item["item"], out, indent + 1)


answers = {}
walk_response(questionnaire_response["item"], answers)

  dWGS/referral_id                              r2026000202
  dWGS/clinical_indication_test_type_id         {'system': 'https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirector
  dWGS/ordering_entity_id                       RTR
  dWGS/glh_laboratory_id                        699N0
  dWGS/retrospective_sample                     New
  dWGS/approved_by                              Duty Scientist on behalf of NEY GMS
  LN/45392-8                                    Allanys
  LN/45394-4                                    Middlesborough
  LN/21112-8                                    1947-04-27
  LN/89061-6                                    9737873971
  dWGS/patient_ngis_id                          p2026000102
  NOS/FamilyStructure                           Duo
  NOS/ParticipantType                           Proband
  dWGS/primary_sample_received_date             2026-08-20T00:00:00+00:00
  dWGS/primary_sample_id_as_received_by_glh     RTR-P0001
  dWGS/primary_sample_id_in_glh_lims      

## Extraction: two mechanisms, side by side

[build.fhir.org/ig/HL7/sdc/en/extraction.html](https://build.fhir.org/ig/HL7/sdc/en/extraction.html)
defines four ways to turn a completed `QuestionnaireResponse` into standalone FHIR
resources — Observation-based, definition-based, template-based, `StructureMap`-based.
`Questionnaire-dWGSSubOrder` itself uses two of them, and which one applies to a given
question is visible directly in the `definition`/`code` printed above:

- **Definition-based**, for almost everything: `item.definition` names the exact target
  element on another resource type — `dWGS/referral_id`'s definition is
  `ServiceRequest#ServiceRequest.requisition`, `LN/21112-8`'s is
  `Patient#Patient.birthDate`. The extractor just needs to know which resource each
  group builds (`Referral` → `ServiceRequest`, `Patient` → `Patient`,
  `PrimarySpecimen`/`DispatchedSpecimen` → `Specimen`) and follow the path.
- **Observation-based**, only for the two `AskAtOrderEntry` questions:
  `NOS/FamilyStructure`'s definition is `Observation#Observation.valueCodeableConcept`,
  *and* it carries its own `item.code`
  (`https://fhir.nwgenomics.nhs.uk/CodeSystem/NWGMSA#FamilyStructure`) — because
  neither `ServiceRequest` nor `Patient` nor `Specimen` has anywhere to put "how many
  people are being tested together," each such question becomes its own standalone
  `Observation`: `Observation.code` from the *question's* code, `Observation.value[x]`
  from the *answer*. That's the one case where extraction produces a whole new
  resource per question rather than filling in a field on one already being built.

In [4]:
for link_id in ("dWGS/referral_id", "LN/21112-8", "NOS/FamilyStructure", "NOS/ParticipantType"):
    d = definitions[link_id]
    print(f"{link_id:24} definition={d['definition']}")
    if d["code"]:
        print(f"{'':24} code={d['code'][0]['system']}#{d['code'][0]['code']}")

dWGS/referral_id         definition=http://hl7.org/fhir/StructureDefinition/ServiceRequest#ServiceRequest.requisition
LN/21112-8               definition=http://hl7.org/fhir/StructureDefinition/Patient#Patient.birthDate
                         code=http://loinc.org#21112-8
NOS/FamilyStructure      definition=http://hl7.org/fhir/StructureDefinition/Observation#Observation.valueCodeableConcept
                         code=https://fhir.nwgenomics.nhs.uk/CodeSystem/NWGMSA#FamilyStructure
NOS/ParticipantType      definition=http://hl7.org/fhir/StructureDefinition/Observation#Observation.valueCodeableConcept
                         code=https://fhir.nwgenomics.nhs.uk/CodeSystem/NWGMSA#ParticipantType


## Building the resources

Same field-by-field approach `08-subcontracted-laboratory-order-from-external-glh.ipynb`
uses — only the source changes: `answers["<linkId>"]` (from the form) instead of
`row["<csv_field>"]` (from the CSV). One difference worth calling out as it comes up:
several `choice`-typed items (`dWGS/clinical_indication_test_type_id`, `LN/66746-9`)
already answer with a full `Coding` (`{system, code, display}`), not just a bare
string — a form-sourced answer can arrive pre-coded in a way a CSV cell never does,
so those slot straight into a resource's `.coding` array with no extra assembly.

### `Patient`

In [5]:
patient_fullurl = f"urn:uuid:{uuid4()}"
patient = {
    "resourceType": "Patient",
    "identifier": [
        {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": answers["LN/89061-6"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": GENOMICS_ENGLAND_ODS}},
         "type": {"coding": [{"system": V2_0203, "code": "PI"}]}, "value": answers["dWGS/patient_ngis_id"]},
    ],
    "name": [{"family": answers["LN/45394-4"], "given": [answers["LN/45392-8"]]}],
    "birthDate": answers["LN/21112-8"],
}
print(json.dumps(patient, indent=2))

{
  "resourceType": "Patient",
  "identifier": [
    {
      "system": "https://fhir.nhs.uk/Id/nhs-number",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "NH"
          }
        ]
      },
      "value": "9737873971"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "8J834"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PI"
          }
        ]
      },
      "value": "p2026000102"
    }
  ],
  "name": [
    {
      "family": "Middlesborough",
      "given": [
        "Allanys"
      ]
    }
  ],
  "birthDate": "1947-04-27"
}


### The two `AskAtOrderEntry` Observations

Coded from the question's own `item.code` — a step further than `08`'s equivalent
Observations, which used `code.text` only ("no NW-GMSA-confirmed coding system exists
for either", per that notebook's own note) since no code existed for the *question* at
the time. It does now: exactly the kind of incremental coding
`10-histocompatibility-immunogenetics-hl7v2-nw-standard.ipynb`'s summary anticipated —
"SNOMED CT/LOINC-code individual 'ask at order' questions later, where a real
requirement calls for it." The *answer* (`"Duo"`, `"Proband"`) is still text-only —
that's a separate, still-unconfirmed coding system per the `Questionnaire`'s own design
note, not something this notebook can invent.

In [6]:
def ask_at_order_observation(link_id, patient_ref):
    d = definitions[link_id]
    question_code = d["code"][0]
    return {
        "resourceType": "Observation",
        "status": "final",
        "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "exam"}]}],
        "code": {"coding": [{"system": question_code["system"], "code": question_code["code"]}], "text": questionnaire_item_text(link_id)},
        "subject": {"reference": patient_ref},
        "valueCodeableConcept": {"text": answers[link_id]},
    }


def questionnaire_item_text(link_id, items=None):
    for item in items or questionnaire["item"]:
        if item["linkId"] == link_id:
            return item["text"]
        if item.get("item"):
            found = questionnaire_item_text(link_id, item["item"])
            if found:
                return found
    return None


family_structure_fullurl = f"urn:uuid:{uuid4()}"
family_structure_observation = ask_at_order_observation("NOS/FamilyStructure", patient_fullurl)

participant_type_fullurl = f"urn:uuid:{uuid4()}"
participant_type_observation = ask_at_order_observation("NOS/ParticipantType", patient_fullurl)

print(json.dumps(family_structure_observation, indent=2))

{
  "resourceType": "Observation",
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "exam"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nwgenomics.nhs.uk/CodeSystem/NWGMSA",
        "code": "FamilyStructure"
      }
    ],
    "text": "Family Structure"
  },
  "subject": {
    "reference": "urn:uuid:c91dae66-ff94-47b4-9d59-55e68f171ee0"
  },
  "valueCodeableConcept": {
    "text": "Duo"
  }
}


### `Specimen`

One `Specimen`, folding the `PrimarySpecimen` and `DispatchedSpecimen` groups' answers
together as multiple identifiers on a single resource — same reasoning `08` uses: both
groups describe the same physical sample at different points in its journey, not two
different specimens.

In [7]:
specimen_fullurl = f"urn:uuid:{uuid4()}"
sample_material_type = answers["LN/66746-9"]  # already a Coding - {system, code, display}

specimen = {
    "resourceType": "Specimen",
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": answers["dWGS/ordering_entity_id"]}},
         "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": answers["dWGS/primary_sample_id_as_received_by_glh"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": answers["dWGS/glh_laboratory_id"]}},
         "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}, "value": answers["dWGS/primary_sample_id_in_glh_lims"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": answers["dWGS/glh_laboratory_id"]}},
         "type": {"coding": [{"system": V2_0203, "code": "STN"}]}, "value": answers["dWGS/glh_sample_consignment_number"]},
    ],
    "type": {"coding": [sample_material_type]},
    "subject": {"reference": patient_fullurl},
    "collection": {
        "collectedDateTime": answers["LN/33882-2"],
        "method": {"text": answers["dWGS/dna_extraction_protocol"]},
        "quantity": {"value": answers["dWGS/dispatched_sample_volume_ul"], "unit": "uL",
                     "system": "http://unitsofmeasure.org", "code": "uL"},
    },
    "receivedTime": answers["dWGS/primary_sample_received_date"],
    "container": [{"identifier": [
        {"type": {"coding": [{"system": V2_0203, "code": "ZCID"}]}, "value": answers["dWGS/dispatched_sample_lsid"]}
    ]}],
}
print(json.dumps(specimen, indent=2))

{
  "resourceType": "Specimen",
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RTR"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "RTR-P0001"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699N0"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "FILL"
          }
        ]
      },
      "value": "YNE26-P0002"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699N0"
        }
      },
      "type": {
        "coding": [


### `ServiceRequest`

`answers["dWGS/clinical_indication_test_type_id"]` is already a `Coding` against the
England Genomic Test Directory - no separate constant/lookup needed, unlike `08`'s CSV
path.

In [8]:
service_request_fullurl = f"urn:uuid:{uuid4()}"
service_request_intent = "reflex-order" if answers["dWGS/retrospective_sample"] == "Retrospective" else "filler-order"

service_request = {
    "resourceType": "ServiceRequest",
    "status": "active",
    "intent": service_request_intent,
    "category": [{"coding": [{"system": "http://snomed.info/sct", "code": "116148004"}]}],
    "code": {"coding": [answers["dWGS/clinical_indication_test_type_id"]]},
    "requisition": {
        "assigner": {"identifier": {"system": ODS_SYSTEM, "value": answers["dWGS/glh_laboratory_id"]}},
        "value": answers["dWGS/referral_id"],
    },
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": answers["dWGS/glh_laboratory_id"]}},
         "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": answers["dWGS/referral_id"]},
    ],
    "subject": {"reference": patient_fullurl},
    "requester": {
        "identifier": {"system": ODS_SYSTEM, "value": answers["dWGS/glh_laboratory_id"]},
        "type": "Organization",
    },
    "specimen": [{"reference": specimen_fullurl, "type": "Specimen"}],
    "supportingInfo": [
        {"reference": family_structure_fullurl, "type": "Observation"},
        {"reference": participant_type_fullurl, "type": "Observation"},
    ],
}
print(json.dumps(service_request, indent=2))

{
  "resourceType": "ServiceRequest",
  "status": "active",
  "intent": "filler-order",
  "category": [
    {
      "coding": [
        {
          "system": "http://snomed.info/sct",
          "code": "116148004"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "R59.1"
      }
    ]
  },
  "requisition": {
    "assigner": {
      "identifier": {
        "system": "https://fhir.nhs.uk/Id/ods-organization-code",
        "value": "699N0"
      }
    },
    "value": "r2026000202"
  },
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699N0"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "r2026000202"
    }
  ],
 

### `MessageHeader` and the `Bundle`

Same shape every earlier notebook's `O21` uses: `sender` is the Order Placer
(`dWGS/glh_laboratory_id` — NE&Y Genomics' own constituent lab, `699N0`),
`destination` is NW Genomics (`699X0`, the RIE).

In [9]:
message_header = {
    "resourceType": "MessageHeader",
    "eventCoding": {"system": "http://terminology.hl7.org/CodeSystem/v2-0003", "code": "O21"},
    "sender": {"identifier": {"system": ODS_SYSTEM, "value": answers["dWGS/glh_laboratory_id"]}},
    "destination": [{"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/RIE",
                      "receiver": {"identifier": {"system": ODS_SYSTEM, "value": NW_GLH_ODS}}}],
    "source": {"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/NEYGenomics", "software": "NE&Y Genomics"},
    "focus": [{"reference": service_request_fullurl}],
}

order_bundle = {
    "resourceType": "Bundle",
    "identifier": {"value": f"urn:uuid:{uuid4()}"},
    "timestamp": datetime.now().astimezone().strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "type": "message",
    "entry": [
        {"fullUrl": f"urn:uuid:{uuid4()}", "resource": message_header},
        {"fullUrl": patient_fullurl, "resource": patient},
        {"fullUrl": specimen_fullurl, "resource": specimen},
        {"fullUrl": family_structure_fullurl, "resource": family_structure_observation},
        {"fullUrl": participant_type_fullurl, "resource": participant_type_observation},
        {"fullUrl": service_request_fullurl, "resource": service_request},
    ],
}

os.makedirs("Output/FHIR/O21", exist_ok=True)
with open("Output/FHIR/O21/dWGS_r2026000202_p2026000102-from-form.json", "w") as f:
    json.dump(order_bundle, f, indent=2)

print("resourceTypes:", [e["resource"]["resourceType"] for e in order_bundle["entry"]])

resourceTypes: ['MessageHeader', 'Patient', 'Specimen', 'Observation', 'Observation', 'ServiceRequest']


### Checking it against `08`'s CSV-built version

Same referral, two different starting points - not expected to be byte-identical (this
notebook's `Specimen` carries `collection.method`/`collection.quantity` from the
`DispatchedSpecimen` answers; the committed fixture doesn't, which looks like drift
from an earlier version of `08`'s own code rather than anything this notebook's
extraction is doing differently) - but the same resource types, for the same patient,
referral, and specimen, either way.

In [10]:
with open("Input/FHIR/O21/dWGS_r2026000202_p2026000102.json") as f:
    csv_built_bundle = json.load(f)

form_types = sorted(e["resource"]["resourceType"] for e in order_bundle["entry"])
csv_types = sorted(e["resource"]["resourceType"] for e in csv_built_bundle["entry"] if e["resource"]["resourceType"] != "RelatedPerson")
print("form-built resourceTypes:", form_types)
print("csv-built resourceTypes :", csv_types)

csv_service_request = next(e["resource"] for e in csv_built_bundle["entry"] if e["resource"]["resourceType"] == "ServiceRequest")
print()
print("referral_id  matches:", service_request["requisition"]["value"] == csv_service_request["requisition"]["value"])
print("test code    matches:", service_request["code"]["coding"][0]["code"] == csv_service_request["code"]["coding"][0]["code"])

form-built resourceTypes: ['MessageHeader', 'Observation', 'Observation', 'Patient', 'ServiceRequest', 'Specimen']
csv-built resourceTypes : ['MessageHeader', 'Observation', 'Observation', 'Patient', 'ServiceRequest', 'Specimen']

referral_id  matches: True
test code    matches: True


## The same process, ending in HL7 v2 `O21` instead

Per the `LAB-1` diagram above, only the *last* step differs by route - everything up to
`answers` (form selection, completion, extraction) is identical whichever wire format
the order eventually leaves as:

- **FHIR-capable EPR → RIE**: build the FHIR `O21` `Bundle` above, send it as a FHIR
  Message; the RIE produces `OML_O21` v2 for the LIMS itself (exactly what `V2_TOOLS`'s
  `/transformToV2` does below - the same conversion the RIE runs internally, per
  `10-histocompatibility-immunogenetics-hl7v2-nw-standard.ipynb`'s note on the RIE's
  FHIR R4 canonical model).
- **EPR with no FHIR capability → RIE**: skip the `Bundle` entirely and hand-build v2
  segments straight from the same `answers` dict -
  `09-genomic-order-management-fhir-to-hl7v2-for-lims.ipynb`'s Step 2 is the fuller
  worked example of exactly that field-by-field `MSH`/`PID`/`ORC`/`OBR` construction,
  just starting from `row`/a FHIR `ServiceRequest` there instead of `answers` here.

Requires a live `V2_TOOLS` (see `.env`) - this cell isn't executed as part of this
notebook's own run.

In [ ]:
r = requests.post(toolsServer + "/transformToV2", data=json.dumps(order_bundle), verify=False,
                   headers={"Content-Type": "application/fhir+json"})
print("HTTP", r.status_code)
print(r.text.replace("\r", "\r\n"))

## Summary

- Order-entry forms in an EPR are, structurally, the same idea HL7's LRI already uses
  for lab *reports* - a `Questionnaire` (a v2 LRI panel's FHIR counterpart) rendered as
  a form, completed as a `QuestionnaireResponse` - see the LHC-Forms link above for the
  closest public renderer to try this against.
- `Questionnaire-dWGSSubOrder` is one of NW-GMSA's own `ServiceRequest.html`-listed
  order forms, and doubles as both a referring Trust's order-entry form and NW
  Genomics' own digital manifest - different groups completed by different
  organisations, at different points in the `LAB-1` flow.
- Pre-population (SDC's `$populate`/continuous population, observation- or
  expression-based) fills in what the completing organisation already knows before a
  human sees the form - which, concretely, is why `Patient` answers can be
  automated but `LaboratoryQC` answers can't be, at referral time.
- Extraction turns the completed form into real resources two ways, both visible
  directly in the `Questionnaire`'s own `item.definition`/`item.code`: definition-based
  for almost everything (the field just goes where `item.definition` says), and
  Observation-based only for the two `AskAtOrderEntry` questions, each becoming its own
  `Observation` coded from the *question*.
- Those resources assemble into the same FHIR Message `O21` `Bundle` shape every
  earlier notebook in this series uses - and, per the `LAB-1` flow, the RIE converts
  that into `OML_O21` v2 for the LIMS the same way it does for a v2-only EPR that skips
  FHIR and sends `O01`/`O21` directly - one extraction pipeline, two possible wire
  formats out.